## Simulation-based validation

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cmdstanpy import CmdStanModel
from pathlib import Path
import scipy.stats as st
import os

from joblib import Parallel, delayed
from tqdm.auto import tqdm
import traceback
import ast

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Dict, Optional

In [1]:
from bdgdm.simulation_validation import (
    simulate_dataset_multi_gene,
    summarize_simulated_truth,
    summarize_simulated_counts,
)

#### Compile Stan model

In [5]:
model_single = CmdStanModel(stan_file="../bdgdm/stan/bdgdm_single.stan")

#### Simulate single-mode dataset

In [7]:
sim_df, truth_df, covars = simulate_dataset_multi_gene(
    G=100,
    N=200,
    subtype_label="all_samples",
    seed=123,
)

In [11]:
sim_df.head()

,gene,scenario,analysis_mode,sample_id,expr,copies,purity,sf,subtype,subtype_idx,dose_log,dev,mu_true
0,G1,null,single_group,S1,81,5.0,0.665606,0.727739,all_samples,1,0.916291,1.5,131.013859
1,G1,null,single_group,S2,422,1.0,0.781787,1.123083,all_samples,1,-0.693147,-0.5,231.349402
2,G1,null,single_group,S3,120,2.0,0.665322,1.255084,all_samples,1,0.000000,0.0,225.800272
3,G1,null,single_group,S4,190,5.0,0.541880,0.926861,all_samples,1,0.916291,1.5,137.818204
4,G1,null,single_group,S5,202,5.0,0.605982,1.077741,all_samples,1,0.916291,1.5,177.750102


In [13]:
truth_df.head()

,gene,scenario,analysis_mode,subtype_s1,phi_true,b_noncancer_log_true,b0_true,b_scaling_true,b_deviation_true,b0_mean_true,b_scaling_mean_true,b_dev_mean_true,b0_s1_true,b_scaling_s1_true,b_deviation_s1_true
0,G1,null,single_group,all_samples,9.283786,2.438495,5.578391,0.042516,-0.025788,5.578391,0.042516,-0.025788,5.578391,0.042516,-0.025788
1,G2,null,single_group,all_samples,9.452169,1.891633,4.950269,-0.009027,-0.038902,4.950269,-0.009027,-0.038902,4.950269,-0.009027,-0.038902
2,G3,null,single_group,all_samples,8.479169,1.566187,4.939703,0.052350,0.068623,4.939703,0.052350,0.068623,4.939703,0.052350,0.068623
3,G4,null,single_group,all_samples,8.945998,1.514079,4.612789,-0.084300,-0.008123,4.612789,-0.084300,-0.008123,4.612789,-0.084300,-0.008123
4,G5,mixed,single_group,all_samples,10.422171,2.405934,5.302279,0.357199,-0.334792,5.302279,0.357199,-0.334792,5.302279,0.357199,-0.334792


In [61]:
# Simulate 200 genes
#if __name__ == "__main__":
#    sim_df, truth_df, covars = simulate_dataset_multi_gene(
#        G=400,
#        N=200,
#        subtype_levels=("A", "B"),
#        seed=123,
#        scenario_probs={
#            "null": 0.30,
#            "scaling": 0.30,
#            "deviation_pos": 0.10,
#            "deviation_neg": 0.10,
#            "mixed": 0.20,
#        },
#    )

In [20]:
 print(truth_df["scenario"].value_counts())

scenario
null             33
scaling          32
mixed            19
deviation_neg    10
deviation_pos     6
Name: count, dtype: int64


In [22]:
print(summarize_simulated_truth(truth_df))

              b0_true                                                    \
                count      mean       std       min    median       max   
scenario                                                                  
deviation_neg      10  4.998766  0.847426  3.965804  5.239537  6.205613   
deviation_pos       6  5.063918  0.690071  4.107754  5.221604  5.893771   
mixed              19  5.199436  0.772976  3.503813  5.380188  6.248219   
null               33  4.932189  0.572974  3.738121  4.907578  6.002447   
scaling            32  4.874432  0.783284  3.384407  4.809113  6.888946   

              b_scaling_true                                ...  \
                       count      mean       std       min  ...   
scenario                                                    ...   
deviation_neg             10 -0.008205  0.042044 -0.053950  ...   
deviation_pos              6  0.008977  0.033576 -0.037556  ...   
mixed                     19  0.454059  0.090917  0.282382  ... 

In [24]:
count_summary = summarize_simulated_counts(sim_df)
print(count_summary.head())

        scenario gene    n  mean_expr      var_expr  zero_frac  mean_mu_true  \
0  deviation_neg  G14  200    146.435   4970.608819        0.0    142.505391   
1  deviation_neg  G18  200    203.295  11030.490427        0.0    197.886333   
2  deviation_neg  G27  200    356.940  32485.252663        0.0    345.837197   
3  deviation_neg  G34  200    229.240  14114.776281        0.0    227.609384   
4  deviation_neg  G36  200    182.010   9766.753668        0.0    179.095107   

    var_mu_true  
0   2001.841154  
1   5123.087658  
2  14069.693292  
3   7544.572075  
4   5228.100189  


#### Run MCMC test

In [44]:
res_mcmc = fit_one_gene_de(
        sim_df,
        model,
        gene="G1",
        engine="nuts",
        chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=123,
        show_progress=False,
        adapt_delta=0.9,
        max_treedepth=12,
        return_ppc=True
    )

16:51:37 - cmdstanpy - INFO - CmdStan start processing
16:51:37 - cmdstanpy - INFO - Chain [1] start processing
16:51:37 - cmdstanpy - INFO - Chain [2] start processing
16:51:37 - cmdstanpy - INFO - Chain [3] start processing
16:51:37 - cmdstanpy - INFO - Chain [4] start processing
17:00:12 - cmdstanpy - INFO - Chain [2] done processing
17:00:39 - cmdstanpy - INFO - Chain [1] done processing
17:00:41 - cmdstanpy - INFO - Chain [3] done processing
17:00:51 - cmdstanpy - INFO - Chain [4] done processing
17:00:51 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 990 iterations at max treedepth (99.0%)
	Chain 2 had 566 iterations at max treedepth (56.6%)
	Chain 3 had 877 iterations at max treedepth (87.7%)
	Chain 4 had 997 iterations at max treedepth (99.7%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [49]:
res_mcmc

{'status': 'ok',
 'gene': 'G1',
 'N': 200,
 'n_aneup': 87,
 'cna': 'all',
 'subtype_levels': ['A', 'B'],
 'tumor0_lfc_median': -0.5836848382953364,
 'tumor0_lfc_q025': -0.7170723100950633,
 'tumor0_lfc_q975': -0.4490948080442763,
 'p_up_tumor': 0.0,
 'lfsr_tumor': 0.0,
 'p_rope_tumor': 0.0,
 'delta_scaling_median': 0.003145505,
 'delta_scaling_q025': -0.432539025,
 'delta_scaling_q975': 0.43473895,
 'p_up_scaling': 0.50575,
 'lfsr_scaling': 0.49424999999999997,
 'delta_dev_median': -0.08531965,
 'delta_dev_q025': -0.434306525,
 'delta_dev_q975': 0.26259922500000005,
 'p_up_dev': 0.31925,
 'lfsr_dev': 0.31925,
 'phi_median': 10.88635,
 'phi_q025': 10.042887499999999,
 'phi_q975': 12.743905,
 'b0_s1_median': 6.300929999999999,
 'b0_s1_q025': 6.23388925,
 'b0_s1_q975': 6.3657604999999995,
 'b_scaling_s1_median': 0.1383765,
 'b_scaling_s1_q025': -0.21613607499999998,
 'b_scaling_s1_q975': 0.510145575,
 'b_deviation_s1_median': 0.367992,
 'b_deviation_s1_q025': 0.06756870250000001,
 'b_devi